In [1]:
import pandas as pd
import numpy as np
import os
from itertools import product
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sn

In [2]:
# 1. Load dataset
df=pd.read_csv('../../dataset/diabetes_dataset.csv')
print("Initial data shape:", df.shape)
print(df.head());

Initial data shape: (100000, 16)
   year  gender   age location  race:AfricanAmerican  race:Asian  \
0  2020  Female  32.0  Alabama                     0           0   
1  2015  Female  29.0  Alabama                     0           1   
2  2015    Male  18.0  Alabama                     0           0   
3  2015    Male  41.0  Alabama                     0           0   
4  2016  Female  52.0  Alabama                     1           0   

   race:Caucasian  race:Hispanic  race:Other  hypertension  heart_disease  \
0               0              0           1             0              0   
1               0              0           0             0              0   
2               0              0           1             0              0   
3               1              0           0             0              0   
4               0              0           0             0              0   

  smoking_history    bmi  hbA1c_level  blood_glucose_level  diabetes  
0           never  27.32

In [3]:
# 2. Drop unneeded columns
df.drop(columns=[
    'year','location',
    'race:AfricanAmerican','race:Asian','race:Caucasian','race:Hispanic','race:Other','smoking_history'
], inplace=True)
print("After dropping columns:", df.shape)
print(df.head());

After dropping columns: (100000, 8)
   gender   age  hypertension  heart_disease    bmi  hbA1c_level  \
0  Female  32.0             0              0  27.32          5.0   
1  Female  29.0             0              0  19.95          5.0   
2    Male  18.0             0              0  23.76          4.8   
3    Male  41.0             0              0  27.32          4.0   
4  Female  52.0             0              0  23.75          6.5   

   blood_glucose_level  diabetes  
0                  100         0  
1                   90         0  
2                  160         0  
3                  159         0  
4                   90         0  


In [4]:
# 3. Encode gender
encoder = LabelEncoder()
df['gender'] = encoder.fit_transform(df['gender'])
print("Label Encoding Mapping:", dict(zip(encoder.classes_, range(len(encoder.classes_)))))

Label Encoding Mapping: {'Female': 0, 'Male': 1, 'Other': 2}


In [5]:
# 4. Drop duplicates & 'Other' gender
df.drop_duplicates(inplace=True)
df = df[df['gender'] != 2]
print("After dropping duplicates & 'Other':", df.shape)

After dropping duplicates & 'Other': (91295, 8)


In [6]:
# 5. Normalize numeric columns
cols_to_normalize = ['age','bmi','hbA1c_level','blood_glucose_level']
scaler = MinMaxScaler()
df[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

In [7]:
## here we will perform the interaction
# Binary feature columns
from itertools import product
df['age_high'] = (df['age'] > 60).astype(int)
df['bmi_obese'] = (df['bmi'] > 23).astype(int)
df['hba1c_diabetes'] = (df['hbA1c_level'] > 6.5).astype(int)
df['glucose_high'] = (df['blood_glucose_level'] > 126).astype(int)

binary_cols = ['age_high', 'heart_disease', 'hba1c_diabetes', 'hypertension', 'bmi_obese', 'gender', 'glucose_high']

# Interaction for hbA1c_level with all others except hbA1c_level and blood_glucose_level
for col in binary_cols:
    if col not in ['hba1c_diabetes', 'glucose_high']:
        for val1, val2 in product([0, 1], repeat=2):
            new_col = f"hbA1c_level({val1}){col}({val2})"
            df[new_col] = ((df['hba1c_diabetes'] == val1) & (df[col] == val2)).astype(int)

df['hbA1c_level(0)_glucose_high(0)'] = ((df['glucose_high'] == 0) & (df['hba1c_diabetes'] == 0)).astype(int)
df['hbA1c_level(0)_glucose_high(1)'] = ((df['glucose_high'] == 0) & (df['hba1c_diabetes'] == 1)).astype(int)
df['hbA1c_level(1)_glucose_high(0)'] = ((df['glucose_high'] == 1) & (df['hba1c_diabetes'] == 0)).astype(int)
df['hbA1c_level(1)_glucose_high(1)'] = ((df['glucose_high'] == 1) & (df['hba1c_diabetes'] == 1)).astype(int)
# Interaction for blood_glucose_level with all others except hbA1c_level and blood_glucose_level
for col in binary_cols:
    if col not in ['hba1c_diabetes', 'glucose_high']:
        for val1, val2 in product([0, 1], repeat=2):
            new_col = f"glucose_high({val1}){col}({val2})"
            df[new_col] = ((df['glucose_high'] == val1) & (df[col] == val2)).astype(int)



df.drop(columns=['age_high','bmi_obese','hba1c_diabetes','glucose_high'],inplace=True)
df

,gender,age,hypertension,heart_disease,bmi,hbA1c_level,blood_glucose_level,diabetes,hbA1c_level(0)age_high(0),hbA1c_level(0)age_high(1),...,glucose_high(1)hypertension(0),glucose_high(1)hypertension(1),glucose_high(0)bmi_obese(0),glucose_high(0)bmi_obese(1),glucose_high(1)bmi_obese(0),glucose_high(1)bmi_obese(1),glucose_high(0)gender(0),glucose_high(0)gender(1),glucose_high(1)gender(0),glucose_high(1)gender(1)
0,0,0.399399,0,0,0.202031,0.272727,0.090909,0,1,0,...,0,0,1,0,0,0,1,0,0,0
1,0,0.361862,0,0,0.116013,0.272727,0.045455,0,1,0,...,0,0,1,0,0,0,1,0,0,0
2,1,0.224224,0,0,0.160481,0.236364,0.363636,0,1,0,...,0,0,1,0,0,0,0,1,0,0
3,1,0.512012,0,0,0.202031,0.090909,0.359091,0,1,0,...,0,0,1,0,0,0,0,1,0,0
4,0,0.649650,0,0,0.160364,0.545455,0.045455,0,1,0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0,0.411912,0,0,0.130719,0.545455,0.045455,0,1,0,...,0,0,1,0,0,0,1,0,0,0
99996,0,1.000000,0,0,0.311041,0.400000,0.090909,0,1,0,...,0,0,1,0,0,0,1,0,0,0
99997,1,0.574575,0,0,0.304739,0.490909,0.354545,0,1,0,...,0,0,1,0,0,0,0,1,0,0
99998,0,0.637137,0,0,0.225023,0.454545,0.340909,0,1,0,...,0,0,1,0,0,0,1,0,0,0


In [8]:
from sklearn.feature_selection import mutual_info_classif
# Filter only interaction columns
interaction_cols = [col for col in df.columns if col.startswith('glucose_high(') or col.startswith('hbA1c_level(')]
y = df['diabetes']
# Compute mutual information
mi_scores = mutual_info_classif(df[interaction_cols], y, discrete_features=True)
# len(mi_scores)
mi_series = pd.Series(mi_scores, index=interaction_cols).sort_values(ascending=False)
top_features = mi_series.head(10).index

print(top_features)

Index(['hbA1c_level(0)hypertension(0)', 'hbA1c_level(0)hypertension(1)',
       'glucose_high(0)hypertension(0)', 'glucose_high(0)hypertension(1)',
       'hbA1c_level(0)heart_disease(0)', 'hbA1c_level(0)heart_disease(1)',
       'glucose_high(0)heart_disease(0)', 'glucose_high(0)heart_disease(1)',
       'hbA1c_level(0)gender(0)', 'glucose_high(0)gender(1)'],
      dtype='object')


In [9]:
df.drop(columns=[col for col in df.columns if col in interaction_cols and col not in top_features],inplace=True)
df

,gender,age,hypertension,heart_disease,bmi,hbA1c_level,blood_glucose_level,diabetes,hbA1c_level(0)heart_disease(0),hbA1c_level(0)heart_disease(1),hbA1c_level(0)hypertension(0),hbA1c_level(0)hypertension(1),hbA1c_level(0)gender(0),glucose_high(0)heart_disease(0),glucose_high(0)heart_disease(1),glucose_high(0)hypertension(0),glucose_high(0)hypertension(1),glucose_high(0)gender(1)
0,0,0.399399,0,0,0.202031,0.272727,0.090909,0,1,0,1,0,1,1,0,1,0,0
1,0,0.361862,0,0,0.116013,0.272727,0.045455,0,1,0,1,0,1,1,0,1,0,0
2,1,0.224224,0,0,0.160481,0.236364,0.363636,0,1,0,1,0,0,1,0,1,0,1
3,1,0.512012,0,0,0.202031,0.090909,0.359091,0,1,0,1,0,0,1,0,1,0,1
4,0,0.649650,0,0,0.160364,0.545455,0.045455,0,1,0,1,0,1,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0,0.411912,0,0,0.130719,0.545455,0.045455,0,1,0,1,0,1,1,0,1,0,0
99996,0,1.000000,0,0,0.311041,0.400000,0.090909,0,1,0,1,0,1,1,0,1,0,0
99997,1,0.574575,0,0,0.304739,0.490909,0.354545,0,1,0,1,0,0,1,0,1,0,1
99998,0,0.637137,0,0,0.225023,0.454545,0.340909,0,1,0,1,0,1,1,0,1,0,0


In [10]:
from imblearn.over_sampling import SMOTE
x=df.drop(columns=['diabetes'])
y=df['diabetes']
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [11]:
# 12. Train KNN with k=13
knn = KNeighborsClassifier(n_neighbors=13, metric='manhattan', weights='distance')
knn.fit(X_train, y_train)

KNeighborsClassifier(metric='manhattan', n_neighbors=13, weights='distance')

In [12]:
# 13. Predict and evaluate
y_pred = knn.predict(X_test)

In [13]:
print("\n=== KNN WITH INTERACTIONS (k=13) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


=== KNN WITH INTERACTIONS (k=13) ===
Accuracy: 0.9107837230954597
Precision: 0.5217391304347826
Recall: 0.8105323411562679
F1-score: 0.6348352387357095

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.92      0.95     16512
           1       0.52      0.81      0.63      1747

    accuracy                           0.91     18259
   macro avg       0.75      0.87      0.79     18259
weighted avg       0.93      0.91      0.92     18259

Confusion Matrix:
 [[15214  1298]
 [  331  1416]]


In [14]:
print("AUC:", roc_auc_score(y_test, knn.predict_proba(X_test)[:, 1]))

AUC: 0.9374154835753872
